<a href="https://colab.research.google.com/github/619carey/ai-ml-coursework-projects/blob/main/Week5_Supervised_Machine_Learning_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Assignment: Building First Supervised Machine Learning Model

## 1. Problem Definition, Target, and Features

* **Dataset:** Telco Customer Churn Dataset
* **Target Variable (Output):** `Churn` (Binary: `1` for churned, `0` for retained)
* **Features (Inputs):** `tenure`, `MonthlyCharges`, `TotalCharges`, `Contract`, `PaymentMethod`, `PaperlessBilling`, `SeniorCitizen`
* **Problem Type:** **Binary Classification**.
  * *Why?* The target variable (`Churn`) is categorical with exactly two distinct classes (Yes/No or 1/0), where the objective is to predict discrete class labels rather than a continuous numerical value.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import files
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Loading Dataset
print("Please upload your 'WA_Fn-UseC_-Telco-Customer-Churn.csv' file if not already loaded:")
try:
    df.head()
    print("Using existing DataFrame in memory.")
except NameError:
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)

# 2. Preprocessing & Feature Selection

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])


features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "PaymentMethod",
    "PaperlessBilling",
    "SeniorCitizen",
]
target = "Churn"

X = df[features].copy()
y = df[target].apply(lambda x: 1 if x == "Yes" else 0)

X = pd.get_dummies(X, drop_first=True)


scaler = StandardScaler()
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
X[num_cols] = scaler.fit_transform(X[num_cols])

# 3. Split Dataset into Training (80%) and Test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape:     {X_test.shape}")

# 4. Logistic Regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

# 5. Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 6. Predictions and Evaluations
models = {"Logistic Regression": log_reg, "Random Forest": rf_model}

for name, model in models.items():
    print(f"\n================ {name} ================")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    print(f"Accuracy:  {acc:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

Please upload your 'WA_Fn-UseC_-Telco-Customer-Churn.csv' file if not already loaded:


Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv

Training set shape: (5625, 10)
Test set shape:     (1407, 10)

================ Logistic Regression ================
Accuracy:  0.7939
ROC-AUC:   0.8319

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.64      0.51      0.57       374

    accuracy                           0.79      1407
   macro avg       0.74      0.70      0.72      1407
weighted avg       0.78      0.79      0.79      1407


================ Random Forest ================
Accuracy:  0.7662
ROC-AUC:   0.8028

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.84      1033
           1       0.57      0.52      0.54       374

    accuracy                           0.77      1407
   macro avg       0.70      0.69      0.69      1407
weighted avg       0.76      0

## 2. Model Performance Comparison & Discussion

### Performance Summary:
* **Logistic Regression:**
  * **Accuracy:** 79.39% | **ROC-AUC:** 0.8319
  * **Summary:** Achieved a higher overall accuracy and stronger ROC-AUC score, with better precision (64%) for the minority churn class.
* **Random Forest Classifier:**
  * **Accuracy:** 76.62% | **ROC-AUC:** 0.8028
  * **Summary:** Provided balanced recall for the churned class by capturing non-linear feature interactions, though slightly lower in overall accuracy than Logistic Regression.

### Discussion:
* In churn prediction problems, **Recall** for the churned class is critical because failing to identify a customer who is about to leave (False Negative) can be costlier than a false alarm (False Positive).
* Although decision trees like Random Forest excel at handling complex feature combinations (e.g., short tenure paired with month-to-month contracts), the linear decision boundaries of **Logistic Regression** proved slightly more effective for this specific subset of features, yielding a stronger baseline and higher overall accuracy.